In [4]:
from openai import OpenAI
from dataclasses import dataclass
import os

@dataclass(frozen=True)

class Provider:

    name:str
    is_free:bool
    env_var:str
    base_url:str |None
    model:str

PROVIDERS=[
    Provider(
        name="OpenAI",
        is_free=False,
        env_var="OPENAI_API_KEY",
        base_url=None,
        model="gpt-4o-mini"
    ),

    Provider(
        
        name="Groq",
        is_free=True,
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b"
    ),

    Provider(
        
        name="Open Router",
        is_free=False,
        env_var="OPENROUTER_API_KEY",
        base_url="https://openrouter.ai/api/v1",
        model="nvidia/llama-nemotron-rerank-vl-1b-v2:free"
    )
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)    
    raise ValueError(f"No valid provider found. Please set one of the following environment variables: {expected}")

def build_client(provider:Provider) ->OpenAI:
    api_key=os.environ.get(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(api_key=api_key, base_url=provider.base_url)

def have_any_key() -> bool:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return True
print("Found a provider key." if have_any_key() else "No provider key found.")


Found a provider key.


In [5]:
def llm_reply(prompt:str) -> str:
    provider=select_provider()
    print(f"Using {provider.name} provider")


    client=build_client(provider)                                  #client is build_client function  

    response=client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{"role":"user", "content":prompt}]
    )

    return response.choices[0].message.content

In [6]:
prompt = "Who was the PM of UK before 2020 ? Answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")

Using Open Router provider
Error: Error code: 400 - nvidia/llama-nemotron-rerank-vl-1b-v2:free is a rerank model and cannot be used with the chat/completions endpoint. Use the /api/v1/rerank endpoint instead.
